# 3. 자료구조 - 스택과 큐 - 워밍업

연습문제에서 쓰인 '동작 순서에 걸린 소소한 테크닉'들을 짧은 문제로 다시 짚는다.
클래스나 함수로 감싸지 않고, **리스트로 직접 구현한 절차적 코드**로 본다.
- 스택: `s.append(x)` 로 넣고 `s.pop()` 으로 꺼낸다.
- 큐: `q.append(x)` 로 넣고 `q.pop(0)` 으로 꺼낸다.

시험 전 머릿속으로 먼저 예측하고, 실행해서 맞는지 확인한다.

## 목록
1. 단락 평가(short-circuit)의 순서
2. `>=` 한 글자가 바꾸는 것 (결합성)
3. 어느 것을 먼저 꺼내나 (비가환 연산)
4. 자릿수 -> 값 (빈칸)
5. 앞에 붙이나 뒤에 붙이나
6. 스택은 순서를 뒤집는다
7. 큐 회전 (맨 앞 -> 맨 뒤)
8. `while n > 0`이 놓치는 값

각 셀은 독립 실행된다(준비 셀 없음).

---
## 1. 단락 평가(short-circuit)의 순서

`or`는 왼쪽이 참이면 오른쪽을 아예 실행하지 않는다.
**빈 리스트** 상태에서 아래 두 줄은 각각 어떻게 될지 예측하라. (정상 출력? 아니면 에러?)

In [ ]:
s = []

# (A) 빈 검사를 먼저
print(len(s) == 0 or s.pop() != '(')

# (B) pop을 먼저
print(s.pop() != '(' or len(s) == 0)

**정답:** (A) -> `True` (왼쪽 `len==0`이 참이라 `s.pop()`은 실행조차 안 됨).
(B) -> **IndexError** (왼쪽 `s.pop()`을 먼저 실행 -> 빈 리스트 pop).

괄호 검사의 `if len(s) == 0 or s.pop() != pairs[ch]` 에서 **빈 검사를 먼저** 둔 이유다. 순서를 바꾸면 빈 스택에서 터진다.

---
## 2. `>=` 한 글자가 바꾸는 것 (결합성)

아래는 중위식을 후위식으로 바꾸는 코드다(연산자 스택 = 리스트).
`5 - 3 - 1`을 넣으면 무엇이 출력되나? 그리고 `>=`를 `>`로 바꾸면?

In [ ]:
expr = "5 - 3 - 1"
prec = {'+': 1, '-': 1, '*': 2, '/': 2}
s = []      # 연산자 스택
out = []    # 출력

for t in expr.split():
    if t in prec:
        while len(s) > 0 and prec[s[-1]] >= prec[t]:   # <- 여기 >=
            out.append(s.pop())
        s.append(t)
    else:
        out.append(t)
while len(s) > 0:
    out.append(s.pop())

print(" ".join(out))

**정답:**
- `>=` (원래): `5 3 - 1 -` -> `(5-3)-1 = 1`  (올바른 좌결합)
- `>` 로 바꾸면: `5 3 1 - -` -> `5-(3-1) = 3`  (틀림)

같은 우선순위끼리 만났을 때 왼쪽 것을 먼저 꺼내야(`>=`) 왼쪽부터 계산된다.

---
## 3. 어느 것을 먼저 꺼내나 (비가환 연산)

아래는 후위식 계산 코드다. `10 3 -` 을 계산하면?
그리고 `b = s.pop()` 과 `a = s.pop()` 두 줄의 순서를 뒤바꾸면 어떤 입력에서 답이 틀릴까?

In [ ]:
expr = "10 3 -"
s = []      # 피연산자 스택

for t in expr.split():
    if t in ('+', '-', '*', '/'):
        b = s.pop()      # 오른쪽 (먼저 꺼냄)
        a = s.pop()      # 왼쪽 (나중 꺼냄)
        if t == '+':
            s.append(a + b)
        elif t == '-':
            s.append(a - b)
        elif t == '*':
            s.append(a * b)
        else:
            s.append(a / b)
    else:
        s.append(int(t))

print(s.pop())

**정답:** `10 3 -` -> `7`. 두 pop 줄을 뒤바꾸면 `3 - 10 = -7` 이 나와 **틀린다.**

단, `10 3 +` 는 순서를 바꿔도 `13`으로 같다. `+`, `*`는 교환법칙이 성립하기 때문.
`-`, `/`처럼 순서가 중요한 연산에서만 버그가 드러난다. 그래서 `b`를 먼저, `a`를 나중에 꺼낸다.

---
## 4. 자릿수 -> 값 (빈칸)

3진수 각 자리가 큐 `[1, 0, 2, 1]`로 들어 있다. 앞에서부터 꺼내 10진수 값을 만든다.
빈칸 `___`에 들어갈 것은?

In [ ]:
q = [1, 0, 2, 1]   # 3진수 자릿수 (앞이 가장 높은 자리)
b = 3
t = 0
while len(q) > 0:
    t = ___ * t + q.pop(0)   # <- 빈칸
print(t)   # 34

**정답:** `b` (또는 `3`).

한 자리 꺼낼 때마다 지금까지의 값에 진법을 곱하고(자리 올림) 새 자릿수를 더한다.
`34 = ((1*3+0)*3+2)*3+1`.

---
## 5. 앞에 붙이나 뒤에 붙이나

10진수를 2진수 문자열로 만들 때, 나머지를 문자열 **앞**에 붙이는 것과 **뒤**에 붙이는 것은 결과가 다르다.
`n=13`일 때 두 출력은?

In [ ]:
n = 13
front = ''
while n > 0:
    front = str(n % 2) + front   # 앞에 붙임
    n //= 2

n = 13
back = ''
while n > 0:
    back = back + str(n % 2)     # 뒤에 붙임
    n //= 2

print(front)   # ?
print(back)    # ?

**정답:** 앞에: `1101` (정답). 뒤에: `1011` (뒤집힘).

나머지는 **낮은 자리부터** 나온다. 앞에 붙이면 나중에 나온 높은 자리가 왼쪽으로 가서,
스택에 쌓았다 꺼낸 것과 같은 효과가 난다.

---
## 6. 스택은 순서를 뒤집는다

리스트를 스택으로 써서 다음을 실행하면 무엇이 출력되나?

In [ ]:
s = []
for ch in "STACK":
    s.append(ch)       # 하나씩 push

out = ""
while len(s) > 0:
    out += s.pop()     # 하나씩 pop
print(out)

**정답:** `KCATS`.

push한 순서의 역순으로 pop된다(LIFO). 전부 넣었다 전부 빼면 뒤집기가 된다.

---
## 7. 큐 회전 (맨 앞 -> 맨 뒤)

리스트를 큐로 써서 `q.append(q.pop(0))` 을 하면 맨 앞 원소가 맨 뒤로 간다. 두 번 실행하면?

In [ ]:
q = [1, 2, 3, 4]
q.append(q.pop(0))
q.append(q.pop(0))
print(q)

**정답:** `[3, 4, 1, 2]`.

1이 뒤로, 그다음 2가 뒤로 간다. 카드 버리기 문제의 '다음 카드를 맨 아래로'가 바로 이 동작이다.

---
## 8. `while n > 0`이 놓치는 값

같은 진법 변환 코드를 `n = 13`과 `n = 0` 으로 각각 실행하면?

In [ ]:
n = 13
s = ''
while n > 0:
    s = str(n % 2) + s
    n //= 2
print(repr(s))   # ?

n = 0
s = ''
while n > 0:
    s = str(n % 2) + s
    n //= 2
print(repr(s))   # ?

**정답:** `n=13` -> `'1101'`. `n=0` -> `''` (빈 문자열).

`while n > 0`이 처음부터 거짓이라 루프를 한 번도 안 돈다.
`n == 0`이면 `'0'`을 따로 처리해야 한다.